# 00 — Audit de la validation groupée

Ce notebook charge et visualise uniquement les artefacts figés de validation groupée. Il ne construit aucun découpage, n'entraîne aucun modèle, ne sélectionne aucune variable et n'accède pas aux données de test du challenge.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qrt_forecasting.v2.folds import load_assignment_artifacts

In [ ]:
def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Impossible de localiser la racine du dépôt.')


repo_root = find_repository_root(Path.cwd().resolve())
assignment_path = repo_root / 'artifacts' / 'folds' / 'v2_grouped_assignment.csv'
manifest_path = repo_root / 'reports' / 'validation' / 'v2_grouped_folds_manifest.json'
assignment, manifest = load_assignment_artifacts(assignment_path, manifest_path)

## Identité des artefacts figés

In [ ]:
pd.Series({
    'selected_splitter': manifest['splitter_selection']['selected_splitter'],
    'data_fingerprint_sha256': manifest['data_fingerprint_sha256'],
    'assignment_sha256': manifest['assignment_sha256'],
    'assignment_rows': len(assignment),
})

## Comparaison structurelle des séparateurs candidats

In [ ]:
audit_rows = []
for audit in manifest['splitter_selection']['candidate_audits']:
    audit_rows.append({
        'splitter': audit['splitter'],
        'valid': audit['valid'],
        'max_relative_row_deviation': audit['max_relative_row_deviation'],
        'max_relative_group_deviation': audit['max_relative_group_deviation'],
        'max_absolute_prevalence_deviation': audit['max_absolute_prevalence_deviation'],
        'worst_balance_deviation': audit['worst_balance_deviation'],
    })
splitter_audit = pd.DataFrame(audit_rows).set_index('splitter')
splitter_audit

In [ ]:
balance_columns = [
    'max_relative_row_deviation',
    'max_relative_group_deviation',
    'max_absolute_prevalence_deviation',
]
ax = splitter_audit[balance_columns].plot.bar(figsize=(10, 5))
ax.set_title('Déséquilibre maximal des folds de validation par séparateur')
ax.set_ylabel('Écart')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()

## Développement et lockbox figé

In [ ]:
partition = pd.DataFrame({
    role: manifest['partition'][role]
    for role in ('development', 'lockbox')
}).T
partition[['n_rows', 'n_groups', 'positive_rate', 'row_fraction', 'group_fraction']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
partition['n_rows'].plot.bar(ax=axes[0], title='Lignes par rôle')
partition['n_groups'].plot.bar(ax=axes[1], title='Groupes TS par rôle')
for axis in axes:
    axis.set_xlabel('')
    axis.tick_params(axis='x', rotation=0)
plt.tight_layout()

## Folds de développement définitifs

In [ ]:
development_folds = pd.DataFrame(
    manifest['partition']['development_fold_audit']['folds']
).set_index('fold_id')
development_folds[['n_rows', 'n_groups', 'n_negative', 'n_positive', 'positive_rate']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
development_folds['n_rows'].plot.bar(ax=axes[0], title='Lignes par fold')
development_folds['n_groups'].plot.bar(ax=axes[1], title='Groupes TS par fold')
development_folds['positive_rate'].plot.bar(ax=axes[2], title='Taux positif par fold')
axes[2].axhline(
    manifest['partition']['development']['positive_rate'],
    color='black',
    linestyle='--',
    label='développement',
)
axes[2].legend()
for axis in axes:
    axis.set_xlabel('fold_id')
    axis.tick_params(axis='x', rotation=0)
plt.tight_layout()

## Conclusion de l'audit

Le séparateur retenu respecte les invariants stricts de groupement et de couverture. Le lockbox est figé par groupes `TS` entiers et ne possède aucun identifiant de fold de développement. Tous les travaux de développement de modèles doivent utiliser uniquement les lignes dont le rôle est `development`.